In [ ]:
print("Hello Rishu")

Hello Rishu


In [ ]:
print("will this show up")

will this show up


In [ ]:
print(will this show up)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (265316685.py, line 1)

In [ ]:
(2+4)

6

In [ ]:
4-2

2

In [ ]:
3*2

6

In [ ]:
#division
120/2

60.0

In [ ]:
12/0

ZeroDivisionError: division by zero

In [ ]:
print("2+3")
print("2*3")

2+3
2*3


In [ ]:
print(2*3)

6


In [ ]:
print(2+15, "3+7")

17 3+7


In [ ]:
print("Rishu is",25+3,"year old")

Rishu is 28 year old


In [ ]:
(3-2)

1

In [ ]:
print("rishu is", 25+3,"year old")

rishu is 28 year old


In [ ]:
print("rishu is 25+3 year old")

rishu is 25+3 year old


In [ ]:
(300/5)

60.0

In [ ]:
print("300/5")

300/5


In [ ]:
print("rishu is good")

NameError: name 'rishu' is not defined

In [4]:

from IPython.display import HTML, display

display(HTML(r"""
<div id="nokia-game" style="
width:100%;max-width:370px;box-sizing:border-box;
margin:8px auto;padding:14px;
background:#26352b;border:6px solid #17231b;
border-radius:28px;text-align:center;
font-family:monospace;color:#d5e8b2;
box-shadow:0 8px 20px #0006;">

<h2 style="letter-spacing:4px;margin:3px;">NOKIA</h2>
<p style="font-size:9px;letter-spacing:3px;margin:3px 0 12px">
CONNECTING PEOPLE</p>

<div style="background:#a8b58b;padding:7px;
border:5px solid #111;border-radius:8px;">

<div style="display:flex;justify-content:space-between;
color:#172719;font-weight:bold;font-size:11px;">
<span>SCORE: <span id="score">0</span></span>
<span>BEST: <span id="best">0</span></span>
</div>

<div style="display:flex;justify-content:space-between;
color:#172719;font-weight:bold;font-size:10px;margin:4px 0;">
<span>LEVEL: <span id="level">1</span></span>
<span id="foodType">NORMAL</span>
</div>

<canvas id="snakeCanvas" width="280" height="280"
tabindex="0" style="
width:100%;max-width:280px;display:block;
margin:auto;image-rendering:pixelated;
outline:none;touch-action:none;"></canvas>

<div id="status" style="color:#172719;
font-weight:bold;min-height:17px;font-size:11px;">
PRESS START TO PLAY</div>
</div>

<div style="margin-top:13px;">
<button id="start" class="game-btn">START</button>
<button id="pause" class="game-btn">PAUSE</button>
<button id="restart" class="game-btn">RESET</button>
</div>

<p style="font-size:10px;margin:12px 0 7px;">
ARROW KEYS / WASD TO MOVE</p>

<div style="display:grid;grid-template-columns:repeat(3,43px);
gap:4px;justify-content:center;">
<span></span>
<button class="direction" data-dir="up">▲</button>
<span></span>
<button class="direction" data-dir="left">◀</button>
<button class="direction" data-dir="down">▼</button>
<button class="direction" data-dir="right">▶</button>
</div>

<p style="font-size:8px;margin:14px 0 0;">
NOKIA RETRO GAMING EXPERIENCE</p>
</div>

<style>
#nokia-game *{box-sizing:border-box}
#nokia-game .game-btn{
padding:8px 12px;background:#a8b58b;
border:2px solid #111;font-weight:bold;
border-radius:7px;cursor:pointer;
font-family:monospace;font-size:11px;margin:2px;
}
#nokia-game .direction{
padding:8px;background:#a8b58b;
border:2px solid #111;border-radius:6px;
font-size:17px;cursor:pointer;
}
#nokia-game button:active{
transform:translateY(2px);
}
</style>

<script>
(()=>{
const root=document.getElementById("nokia-game");
if(root.dataset.ready)return;
root.dataset.ready="yes";

const canvas=root.querySelector("#snakeCanvas");
const ctx=canvas.getContext("2d");

const scoreEl=root.querySelector("#score");
const bestEl=root.querySelector("#best");
const levelEl=root.querySelector("#level");
const foodEl=root.querySelector("#foodType");
const status=root.querySelector("#status");

const size=14,cells=20;

let snake,food,bonus,obstacles;
let direction="right",nextDirection="right";
let score=0,level=1,best=0;
let running=false,paused=false,gameOver=false;
let timer=null,bonusTimer=null,audio=null;

try{
best=Number(localStorage.getItem("nokiaSnakeBest")||0);
}catch(e){}
bestEl.textContent=best;

function sound(freq=450,duration=.08){
try{
audio=audio||new(window.AudioContext||window.webkitAudioContext)();
if(audio.state==="suspended")audio.resume();
const osc=audio.createOscillator();
const gain=audio.createGain();
osc.type="square";
osc.frequency.value=freq;
gain.gain.setValueAtTime(.035,audio.currentTime);
gain.gain.exponentialRampToValueAtTime(.001,audio.currentTime+duration);
osc.connect(gain);
gain.connect(audio.destination);
osc.start();
osc.stop(audio.currentTime+duration);
}catch(e){}
}

function randomFree(){
let free=[];
for(let y=0;y<cells;y++){
for(let x=0;x<cells;x++){
if(!snake.some(p=>p.x===x&&p.y===y)&&
!(obstacles||[]).some(p=>p.x===x&&p.y===y)){
free.push({x,y});
}
}}
return free.length?free[Math.floor(Math.random()*free.length)]:null;
}

function placeFood(){
food=randomFree();
if(!food)endGame(true);
}

function placeBonus(){
if(!running||gameOver)return;
bonus=randomFree();
if(bonus){
foodEl.textContent="BONUS";
clearTimeout(bonusTimer);
bonusTimer=setTimeout(()=>{
bonus=null;
foodEl.textContent="NORMAL";
draw();
},7000);
}
}

function draw(){
ctx.fillStyle="#a8b58b";
ctx.fillRect(0,0,280,280);

// LCD grid
ctx.strokeStyle="rgba(35,60,30,.10)";
ctx.lineWidth=.5;
for(let i=0;i<=280;i+=14){
ctx.beginPath();
ctx.moveTo(i,0);ctx.lineTo(i,280);ctx.stroke();
ctx.beginPath();
ctx.moveTo(0,i);ctx.lineTo(280,i);ctx.stroke();
}

// Obstacles
(obstacles||[]).forEach(p=>{
ctx.fillStyle="#172d18";
ctx.fillRect(p.x*size+1,p.y*size+1,12,12);
});

// Normal food
if(food){
ctx.fillStyle="#203c20";
ctx.fillRect(food.x*size+3,food.y*size+3,8,8);
ctx.fillRect(food.x*size+5,food.y*size,4,3);
}

// Bonus food
if(bonus){
ctx.fillStyle="#182b15";
ctx.fillRect(bonus.x*size+2,bonus.y*size+2,10,10);
ctx.fillStyle="#d5e8b2";
ctx.font="bold 10px monospace";
ctx.textAlign="center";
ctx.fillText("+",bonus.x*size+7,bonus.y*size+11);
}

// Snake
snake.forEach((p,i)=>{
ctx.fillStyle=i===0?"#102a16":"#29482a";
ctx.fillRect(p.x*size+1,p.y*size+1,12,12);

if(i===0){
ctx.fillStyle="#b7c99a";
if(direction==="right"||direction==="left"){
let ex=direction==="right"?9:3;
ctx.fillRect(p.x*size+ex,p.y*size+3,2,2);
ctx.fillRect(p.x*size+ex,p.y*size+9,2,2);
}else{
let ey=direction==="down"?9:3;
ctx.fillRect(p.x*size+3,p.y*size+ey,2,2);
ctx.fillRect(p.x*size+9,p.y*size+ey,2,2);
}
}
});

if(!running&&!gameOver&&!paused)overlay("SNAKE GAME","PRESS START");
if(paused)overlay("PAUSED","PRESS START TO RESUME");
if(gameOver)overlay("GAME OVER","SCORE: "+score);
}

function overlay(title,subtitle){
ctx.fillStyle="rgba(20,40,20,.85)";
ctx.fillRect(28,105,224,70);
ctx.fillStyle="#d5e8b2";
ctx.textAlign="center";
ctx.font="bold 20px monospace";
ctx.fillText(title,140,137);
ctx.font="12px monospace";
ctx.fillText(subtitle,140,158);
}

function updateHUD(){
scoreEl.textContent=score;
bestEl.textContent=best;
levelEl.textContent=level;
}

function runLoop(){
clearInterval(timer);
timer=setInterval(update,Math.max(65,145-(level-1)*12));
}

function startGame(){
clearInterval(timer);
clearTimeout(bonusTimer);

snake=[{x:9,y:10},{x:8,y:10},{x:7,y:10}];
direction="right";
nextDirection="right";
score=0;level=1;
bonus=null;obstacles=[];
running=true;paused=false;gameOver=false;
foodEl.textContent="NORMAL";
placeFood();
updateHUD();
status.textContent="GAME RUNNING";
draw();
runLoop();
sound(500,.1);
canvas.focus();
}

function update(){
if(!running)return;

direction=nextDirection;
let head={...snake[0]};

if(direction==="right")head.x++;
if(direction==="left")head.x--;
if(direction==="up")head.y--;
if(direction==="down")head.y++;

let eating=head.x===food.x&&head.y===food.y;
let bonusEating=bonus&&head.x===bonus.x&&head.y===bonus.y;
let body=(eating||bonusEating)?snake:snake.slice(0,-1);

if(head.x<0||head.x>=cells||head.y<0||head.y>=cells||
body.some(p=>p.x===head.x&&p.y===head.y)||
obstacles.some(p=>p.x===head.x&&p.y===head.y)){
endGame();
return;
}

snake.unshift(head);

if(eating||bonusEating){
score+=bonusEating?30:10;
sound(bonusEating?1000:750,.1);

if(bonusEating){
bonus=null;
clearTimeout(bonusTimer);
foodEl.textContent="NORMAL";
}else placeFood();

let newLevel=Math.floor(score/50)+1;
if(newLevel>level){
level=newLevel;
for(let i=0;i<Math.min(2,level-1);i++){
let p=randomFree();
if(p)obstacles.push(p);
}
sound(900,.15);
status.textContent="LEVEL "+level+"!";
}
updateHUD();
runLoop();
if(score>0&&score%50===0)placeBonus();
}else snake.pop();

draw();
}

function endGame(won=false){
running=false;gameOver=true;
clearInterval(timer);clearTimeout(bonusTimer);
sound(won?950:150,.3);

if(score>best){
best=score;
try{localStorage.setItem("nokiaSnakeBest",best);}catch(e){}
}
updateHUD();
status.textContent=won?"YOU WIN!":"GAME OVER! PRESS START";
draw();
}

function changeDirection(d){
if(!running)return;
const opposite={up:"down",down:"up",left:"right",right:"left"};
if(opposite[d]!==direction)nextDirection=d;
}

function togglePause(){
if(gameOver||!snake)return;
if(!running&&!paused)return;
paused=!paused;
running=!paused;
if(paused){
clearInterval(timer);
status.textContent="GAME PAUSED";
}else{
status.textContent="GAME RUNNING";
runLoop();
}
draw();
}

const keys={
ArrowUp:"up",w:"up",W:"up",
ArrowDown:"down",s:"down",S:"down",
ArrowLeft:"left",a:"left",A:"left",
ArrowRight:"right",d:"right",D:"right"
};

document.addEventListener("keydown",e=>{
if(!root.contains(document.activeElement))return;
if(keys[e.key]){
e.preventDefault();
changeDirection(keys[e.key]);
}
if(e.key===" "){
e.preventDefault();
togglePause();
}
});

root.querySelectorAll(".direction").forEach(btn=>{
btn.onclick=()=>changeDirection(btn.dataset.dir);
});

root.querySelector("#start").onclick=()=>{
if(paused)togglePause();
else startGame();
};
root.querySelector("#pause").onclick=togglePause;
root.querySelector("#restart").onclick=startGame;

let touch=null;
canvas.addEventListener("touchstart",e=>{
let t=e.touches[0];
touch={x:t.clientX,y:t.clientY};
},{passive:true});

canvas.addEventListener("touchend",e=>{
if(!touch)return;
let t=e.changedTouches[0];
let dx=t.clientX-touch.x,dy=t.clientY-touch.y;
if(Math.max(Math.abs(dx),Math.abs(dy))>20){
changeDirection(Math.abs(dx)>Math.abs(dy)?
(dx>0?"right":"left"):(dy>0?"down":"up"));
}
touch=null;
},{passive:true});

snake=[{x:9,y:10},{x:8,y:10},{x:7,y:10}];
food={x:14,y:10};
bonus=null;obstacles=[];
updateHUD();
draw();
})();
</script>
"""))
